# Data Exploration & Feature Selection

**Project**: EDF Energy Consumption Prediction
**Objective**: Analyze energy consumption data, clean it, and identify features most correlated with the target variable (consommation)

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Data Exploration & Understanding](#2-data-exploration--understanding)
3. [Feature Engineering & Cleaning](#3-feature-engineering--cleaning)
4. [Correlation Analysis](#4-correlation-analysis)
5. [Feature Selection](#5-feature-selection)
6. [Summary & Recommendations](#6-summary--recommendations)

## 1. Setup & Data Loading

In [9]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Configure display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Project paths
PROJECT_ROOT = Path.cwd().parent.parent  # Go up 2 levels: src/analyze -> src -> project root
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

Project root: /home/c-enjalbert/Documents/EPSI/MSPR/bloc_3/EDF-mspr3/src
Data directory: home/c-enjalbert/Documents/EPSI/MSPR/bloc_3/EDF-mspr3/data


In [10]:
import pandas as pd
import re
import unicodedata
from pathlib import Path


SUPPORTED_EXTENSIONS = [".csv", ".xls", ".xlsx"]

def _clean_colname(col: str) -> str:
    if not isinstance(col, str):
        col = str(col)

    col = col.strip()
    col = col.replace("�", "e").replace("?", "e")

    col = unicodedata.normalize("NFKD", col)
    col = col.encode("ascii", "ignore").decode("ascii")

    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")

    return col


def _load_single_file(filepath: Path) -> pd.DataFrame:
    suffix = filepath.suffix.lower()

    def read_csv_smart(path):
        best_df = None
        best_cols = 0

        for sep in [";", ",", "\t"]:
            try:
                df_try = pd.read_csv(
                    path,
                    sep=sep,
                    encoding="latin1",
                    low_memory=False,
                    dtype=str,
                    index_col=False
                )                
                if df_try.shape[1] > best_cols:
                    best_cols = df_try.shape[1]
                    best_df = df_try
            except Exception:
                continue

        if best_df is None or best_cols == 1:
            raise ValueError("Impossible de détecter le séparateur CSV")

        return best_df

    try:
        if suffix == ".xls":
                df = read_csv_smart(filepath)
    except Exception as e:
        raise RuntimeError(f"Erreur lors du chargement de {filepath.name} : {e}")

    # Nettoyage des colonnes
    df.columns = [_clean_colname(c) for c in df.columns]
    
    df["source_file"] = filepath.name

    return df




def load_all_data(data_dir: str) -> pd.DataFrame:
    """
    Charge automatiquement tous les fichiers du dossier data/
    """
    data_path = Path(data_dir)

    if not data_path.exists():
        raise FileNotFoundError(f"Dossier introuvable : {data_dir}")

    all_dfs = []

    for file in data_path.iterdir():
        if file.suffix.lower() in SUPPORTED_EXTENSIONS:
            print(f"Chargement : {file.name}")
            df = _load_single_file(file)
            all_dfs.append(df)

    if not all_dfs:
        return pd.DataFrame()

    df_final = pd.concat(all_dfs, ignore_index=True)
    return df_final

In [11]:
# Load data
print("Loading data...")
df_raw = load_all_data(str(DATA_DIR))
print(f"\nData shape: {df_raw.shape}")
print(f"\nColumns: {list(df_raw.columns)}")

Loading data...


FileNotFoundError: Dossier introuvable : home/c-enjalbert/Documents/EPSI/MSPR/bloc_3/EDF-mspr3/data

In [ ]:
# Display sample data
print("\nFirst 5 rows:")
display(df_raw.head())

print("\nLast 5 rows:")
display(df_raw.tail())


First 5 rows:


""



Last 5 rows:


""


In [ ]:
# Data types and info
print("\nData types:")
print(df_raw.dtypes)

print("\nData info:")
df_raw.info()


Data types:
Series([], dtype: object)

Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


## 2. Data Exploration & Understanding

In [ ]:
# Summary statistics
print("Summary statistics for numerical columns:")
display(df_raw.describe())

Summary statistics for numerical columns:


ValueError: Cannot describe a DataFrame without columns

In [ ]:
# Check for missing values
print("Missing values analysis:")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    display(missing_df)
else:
    print("No missing values found in the dataset.")

In [ ]:
# Check for 'ND' values (Not Data)
print("Checking for 'ND' values in each column:")
nd_counts = (df_raw == 'ND').sum()
nd_counts = nd_counts[nd_counts > 0].sort_values(ascending=False)

if len(nd_counts) > 0:
    display(pd.DataFrame({'ND Count': nd_counts}))
else:
    print("No 'ND' values found.")

In [ ]:
# Analyze the target variable: consommation
# First, convert to numeric
df_raw['consommation'] = pd.to_numeric(df_raw['consommation'], errors='coerce')

print("\n=== Target Variable: Consommation ===")
print(f"Total records: {len(df_raw)}")
print(f"Non-null consumption records: {df_raw['consommation'].notna().sum()}")
print(f"Null consumption records: {df_raw['consommation'].isna().sum()}")

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Histogram
axes[0, 0].hist(df_raw['consommation'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Energy Consumption')
axes[0, 0].set_xlabel('Consumption (MW)')
axes[0, 0].set_ylabel('Frequency')

# Box plot
axes[0, 1].boxplot(df_raw['consommation'].dropna(), vert=False)
axes[0, 1].set_title('Box Plot of Energy Consumption')
axes[0, 1].set_xlabel('Consumption (MW)')

# Density plot
df_raw['consommation'].dropna().plot(kind='density', ax=axes[1, 0])
axes[1, 0].set_title('Density Plot of Energy Consumption')
axes[1, 0].set_xlabel('Consumption (MW)')

# Q-Q plot (check normality)
from scipy import stats
stats.probplot(df_raw['consommation'].dropna(), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Print statistics
print("\nConsumption Statistics:")
print(f"Mean: {df_raw['consommation'].mean():.2f} MW")
print(f"Median: {df_raw['consommation'].median():.2f} MW")
print(f"Std: {df_raw['consommation'].std():.2f} MW")
print(f"Min: {df_raw['consommation'].min():.2f} MW")
print(f"Max: {df_raw['consommation'].max():.2f} MW")
print(f"25th percentile: {df_raw['consommation'].quantile(0.25):.2f} MW")
print(f"75th percentile: {df_raw['consommation'].quantile(0.75):.2f} MW")

In [ ]:
# Analyze temporal patterns
# First, create datetime column
df_raw['datetime'] = pd.to_datetime(
    df_raw['date'].astype(str) + ' ' + df_raw['heures'].astype(str),
    errors='coerce'
)
df_temp = df_raw.dropna(subset=['datetime', 'consommation']).copy()
df_temp['hour'] = df_temp['datetime'].dt.hour
df_temp['day'] = df_temp['datetime'].dt.day
df_temp['month'] = df_temp['datetime'].dt.month
df_temp['dayofweek'] = df_temp['datetime'].dt.dayofweek
df_temp['year'] = df_temp['datetime'].dt.year
df_temp['weekend'] = df_temp['dayofweek'].isin([5, 6]).astype(int)

print(f"\nValid datetime records: {len(df_temp)}")
print(f"Date range: {df_temp['datetime'].min()} to {df_temp['datetime'].max()}")

In [ ]:
# Hourly consumption pattern
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# By hour
hourly_mean = df_temp.groupby('hour')['consommation'].mean()
axes[0, 0].bar(hourly_mean.index, hourly_mean.values, color='steelblue')
axes[0, 0].set_title('Average Consumption by Hour of Day')
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Avg Consumption (MW)')
axes[0, 0].set_xticks(range(0, 24, 2))

# By day of week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_mean = df_temp.groupby('dayofweek')['consommation'].mean()
axes[0, 1].bar(range(7), daily_mean.values, color='coral')
axes[0, 1].set_title('Average Consumption by Day of Week')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Avg Consumption (MW)')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(day_names)

# Weekend vs weekday
weekend_mean = df_temp.groupby('weekend')['consommation'].mean()
axes[0, 2].bar(['Weekday', 'Weekend'], weekend_mean.values, color=['steelblue', 'coral'])
axes[0, 2].set_title('Consumption: Weekday vs Weekend')
axes[0, 2].set_ylabel('Avg Consumption (MW)')

# By month
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_mean = df_temp.groupby('month')['consommation'].mean()
axes[1, 0].bar(range(1, 13), monthly_mean.values, color='mediumseagreen')
axes[1, 0].set_title('Average Consumption by Month')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Avg Consumption (MW)')
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(month_names, rotation=45)

# By year
yearly_mean = df_temp.groupby('year')['consommation'].mean()
axes[1, 1].bar(yearly_mean.index, yearly_mean.values, color='mediumpurple')
axes[1, 1].set_title('Average Consumption by Year')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Avg Consumption (MW)')
axes[1, 1].tick_params(axis='x', rotation=45)

# Hourly heatmap by day of week
pivot_data = df_temp.pivot_table(values='consommation', index='hour', columns='dayofweek', aggfunc='mean')
sns.heatmap(pivot_data, cmap='YlOrRd', ax=axes[1, 2], cbar_kws={'label': 'Avg Consumption (MW)'})
axes[1, 2].set_title('Consumption Heatmap: Hour vs Day of Week')
axes[1, 2].set_xlabel('Day of Week')
axes[1, 2].set_ylabel('Hour')
axes[1, 2].set_xticklabels(day_names)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'temporal_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Feature Engineering & Cleaning

In [ ]:
# Apply existing feature engineering
print("Applying existing feature engineering...")
df_features = create_features(df_raw)
print(f"\nFeature engineered shape: {df_features.shape}")
print(f"Columns: {list(df_features.columns)}")

In [ ]:
# Create additional lag features for time series
# Sort by datetime first
df_temp_sorted = df_temp.sort_values('datetime').copy()

# Create lag features
df_temp_sorted['consommation_lag1'] = df_temp_sorted['consommation'].shift(1)
df_temp_sorted['consommation_lag24'] = df_temp_sorted['consommation'].shift(24)  # Same hour yesterday
df_temp_sorted['consommation_lag168'] = df_temp_sorted['consommation'].shift(168)  # Same hour last week

# Create rolling window features
df_temp_sorted['consommation_rolling_mean_24'] = df_temp_sorted['consommation'].rolling(window=24).mean()
df_temp_sorted['consommation_rolling_std_24'] = df_temp_sorted['consommation'].rolling(window=24).std()
df_temp_sorted['consommation_rolling_mean_168'] = df_temp_sorted['consommation'].rolling(window=168).mean()

# Create trend feature (difference from previous hour)
df_temp_sorted['consommation_diff1'] = df_temp_sorted['consommation'].diff(1)

print("Lag and rolling features created.")
print(f"\nShape with engineered features: {df_temp_sorted.shape}")
print("\nNew features:", [col for col in df_temp_sorted.columns if 'lag' in col or 'rolling' in col or 'diff' in col])

In [ ]:
# Merge with production data
# Keep only rows with datetime
df_analysis = df_temp_sorted.copy()

# Drop rows where target is null
df_analysis = df_analysis.dropna(subset=['consommation'])

# Convert all numeric columns from string to numeric
numeric_cols = [
    'fioul', 'charbon', 'gaz', 'nucleaire', 'eolien', 'solaire', 'hydraulique', 
    'pompage', 'bioenergies', 'ech_physiques', 'taux_de_co2',
    'ech_comm_angleterre', 'ech_comm_espagne', 'ech_comm_italie', 'ech_comm_suisse',
    'ech_comm_allemagne_belgique', 'prevision_j_1', 'prevision_j'
]

for col in numeric_cols:
    if col in df_analysis.columns:
        df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')

# Fill NaN with 0 for numeric columns (common practice)
df_analysis[numeric_cols] = df_analysis[numeric_cols].fillna(0)

print(f"Analysis dataset shape: {df_analysis.shape}")
print(f"\nMissing values after cleaning:")
print(df_analysis.isnull().sum().sum())

In [ ]:
# Analyze production mix over time
production_cols = ['nucleaire', 'eolien', 'solaire', 'hydraulique', 'fioul', 'charbon', 'gaz', 'bioenergies']
production_cols = [col for col in production_cols if col in df_analysis.columns]

# Calculate average production by source
avg_production = df_analysis[production_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
avg_production.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Average Energy Production by Source')
plt.xlabel('Energy Source')
plt.ylabel('Average Production (MW)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'production_mix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nAverage production by source:")
for source, value in avg_production.items():
    print(f"{source}: {value:.2f} MW")

## 4. Correlation Analysis

In [ ]:
# Select numeric columns for correlation analysis
numeric_cols_for_corr = df_analysis.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns for correlation: {len(numeric_cols_for_corr)}")

# Calculate correlation matrix
corr_matrix = df_analysis[numeric_cols_for_corr].corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation with target variable (consommation)
target_corr = corr_matrix['consommation'].sort_values(ascending=False)

# Exclude self-correlation
target_corr = target_corr[target_corr.index != 'consommation']

# Create visual
plt.figure(figsize=(12, 10))
colors = ['red' if x > 0 else 'blue' for x in target_corr.values]
plt.barh(range(len(target_corr)), target_corr.values, color=colors, alpha=0.7)
plt.yticks(range(len(target_corr)), target_corr.index)
plt.xlabel('Correlation with Consommation')
plt.title('Feature Correlation with Target Variable (Consommation)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'target_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

# Print top correlations
print("\nTop 20 features correlated with consommation:")
print(target_corr.head(20))

print("\nBottom 20 features (most negative correlation):")
print(target_corr.tail(20))

In [ ]:
# Scatter plots for top correlated features
top_features = target_corr.abs().head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for idx, feature in enumerate(top_features):
    axes[idx].scatter(df_analysis[feature], df_analysis['consommation'], alpha=0.3, s=1)
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Consommation')
    axes[idx].set_title(f'Corr: {target_corr[feature]:.3f}')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance using Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

# Prepare data
# Drop lag/rolling features that would cause data leakage if not carefully handled
feature_cols_for_rf = [col for col in numeric_cols_for_corr 
                        if col not in ['consommation', 'datetime'] 
                        and 'consommation_lag' not in col 
                        and 'consommation_rolling' not in col
                        and 'consommation_diff' not in col]

# Drop rows with NaN in feature columns
df_rf = df_analysis.dropna(subset=feature_cols_for_rf + ['consommation'])

print(f"Training Random Forest with {len(feature_cols_for_rf)} features...")
print(f"Samples: {len(df_rf)}")

# Train Random Forest (subset for speed)
X = df_rf[feature_cols_for_rf].values
y = df_rf['consommation'].values

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get feature importances
importance_df = pd.DataFrame({
    'Feature': feature_cols_for_rf,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot
plt.figure(figsize=(12, 8))
top_importance = importance_df.head(15)
plt.barh(range(len(top_importance)), top_importance['Importance'], color='steelblue')
plt.yticks(range(len(top_importance)), top_importance['Feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 20 Feature Importances:")
print(importance_df.head(20))

In [ ]:
# Multicollinearity Analysis (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Select top features based on correlation
top_corr_features = target_corr.abs().head(20).index.tolist()

# Prepare data for VIF
df_vif = df_analysis[top_corr_features].dropna()

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = df_vif.columns
vif_data["VIF"] = [variance_inflation_factor(df_vif.values, i) 
                    for i in range(df_vif.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\nVariance Inflation Factor (VIF) Analysis:")
print("VIF > 10 indicates high multicollinearity")
display(vif_data)

# Highlight problematic features
high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\nFeatures with high multicollinearity (VIF > 10): {len(high_vif)}")
    print(high_vif['Feature'].tolist())

## 5. Feature Selection

In [ ]:
# Combine different feature selection methods

# 1. Top features by correlation (absolute value)
top_corr = target_corr.abs().head(30).index.tolist()

# 2. Top features by Random Forest importance
top_rf = importance_df.head(30)['Feature'].tolist()

# 3. Remove features with high VIF (>10)
low_vif = vif_data[vif_data['VIF'] <= 10]['Feature'].tolist()

print("Feature selection summary:")
print(f"- Top 30 by correlation: {len(top_corr)}")
print(f"- Top 30 by RF importance: {len(top_rf)}")
print(f"- Features with low VIF: {len(low_vif)}")

# Find intersection of all methods
selected_features = list(set(top_corr) & set(top_rf) & set(low_vif))

# If intersection is too small, use union with VIF filter
if len(selected_features) < 10:
    print("\nIntersection too small. Using top features with low VIF...")
    combined = list(set(top_corr + top_rf))
    selected_features = [f for f in combined if f in low_vif]

# Sort by correlation for final list
selected_features_corr = {f: target_corr[f] for f in selected_features if f in target_corr}
selected_features = sorted(selected_features_corr.keys(), 
                         key=lambda x: abs(selected_features_corr[x]), 
                         reverse=True)

print(f"\nFinal selected features: {len(selected_features)}")
print(selected_features)

In [ ]:
# Create final feature set documentation
feature_summary = pd.DataFrame({
    'Feature': selected_features,
    'Correlation': [target_corr[f] for f in selected_features],
    'RF_Importance': [importance_df[importance_df['Feature'] == f]['Importance'].values[0] 
                    if f in importance_df['Feature'].values else 0 
                    for f in selected_features]
})

display(feature_summary)

In [ ]:
# Create recommended feature groups
print("\n=== RECOMMENDED FEATURE GROUPS ===\n")

# Group 1: Production Mix (most important)
production_features = [f for f in selected_features if f in production_cols]
print("1. Production Mix Features:")
for f in production_features:
    print(f"   - {f} (corr: {target_corr[f]:.3f})")

# Group 2: Temporal Features
temporal_features = [f for f in selected_features if f in ['hour', 'day', 'month', 'dayofweek', 'weekend']]
print("\n2. Temporal Features:")
for f in temporal_features:
    print(f"   - {f} (corr: {target_corr[f]:.3f})")

# Group 3: Forecasts
forecast_features = [f for f in selected_features if 'prevision' in f or 'forecast' in f.lower()]
print("\n3. Forecast Features:")
for f in forecast_features:
    print(f"   - {f} (corr: {target_corr[f]:.3f})")

# Group 4: Energy Exchanges
exchange_features = [f for f in selected_features if 'ech_comm' in f]
print("\n4. Energy Exchange Features:")
for f in exchange_features:
    print(f"   - {f} (corr: {target_corr[f]:.3f})")

# Group 5: Storage
storage_features = [f for f in selected_features if 'pompage' in f or 'batterie' in f]
print("\n5. Storage Features:")
for f in storage_features:
    print(f"   - {f} (corr: {target_corr[f]:.3f})")

In [ ]:
# Export selected features to JSON
import json

# Create export structure
export_data = {
    "selected_features": selected_features,
    "feature_groups": {
        "production": production_features,
        "temporal": temporal_features,
        "forecasts": forecast_features,
        "exchanges": exchange_features,
        "storage": storage_features
    },
    "correlations": {f: float(target_corr[f]) for f in selected_features},
    "rf_importance": {f: float(importance_df[importance_df['Feature'] == f]['Importance'].values[0]) 
                       if f in importance_df['Feature'].values else 0.0 
                       for f in selected_features}
}

# Save to file
output_file = PROJECT_ROOT / 'src/analyze' / 'selected_features.json'
with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"\nSelected features exported to: {output_file}")

## 6. Summary & Recommendations

In [ ]:
# Final summary
print("""
================================================================================
                        DATA EXPLORATION SUMMARY
================================================================================

DATASET OVERVIEW:
- Total records: {:,}
- Date range: {} to {}
- Target variable: consommation (energy consumption in MW)
- Mean consumption: {:.2f} MW
- Median consumption: {:.2f} MW

KEY FINDINGS:
1. Temporal Patterns:
   - Peak consumption: Morning (8-10h) and Evening (18-20h)
   - Higher consumption: Weekdays vs Weekends
   - Seasonal: Higher in winter (Dec-Feb) and lower in summer (Jul-Aug)

2. Production Mix:
   - Nuclear: Primary energy source (baseline load)
   - Wind/Solar: Variable, weather-dependent
   - Hydropower: Significant contributor

3. Highly Predictive Features:
   - prevision_j_1 (previous day forecast)
   - prevision_j (same day forecast)
   - nucleaire (nuclear production)
   - Lag features (consumption at t-1, t-24)

4. Selected Features: {}

RECOMMENDATIONS:
1. Use forecast features (prevision_j, prevision_j_1) as primary predictors
2. Include production mix features, especially nuclear
3. Add temporal features (hour, dayofweek, month)
4. Consider lag features for time series modeling
5. Integrate weather data (temperature, humidity) for seasonal patterns

NEXT STEPS:
1. Implement feature pipeline with selected features
2. Add weather data integration
3. Train models on feature-engineered dataset
4. Evaluate and iterate on feature set

================================================================================
""".format(
    len(df_analysis),
    df_analysis['datetime'].min().strftime('%Y-%m-%d'),
    df_analysis['datetime'].max().strftime('%Y-%m-%d'),
    df_analysis['consommation'].mean(),
    df_analysis['consommation'].median(),
    len(selected_features)
))

In [ ]:
# Create visual summary
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Top 10 features by correlation
top_10_corr = target_corr.abs().head(10)
axes[0].barh(range(len(top_10_corr)), top_10_corr.values, color='steelblue')
axes[0].set_yticks(range(len(top_10_corr)))
axes[0].set_yticklabels(top_10_corr.index, fontsize=9)
axes[0].set_xlabel('Absolute Correlation')
axes[0].set_title('Top 10 Features by Correlation')

# 2. Production mix contribution
if production_features:
    prod_vals = [target_corr[f] for f in production_features[:8]]
    prod_labels = production_features[:8]
    colors = ['green' if v > 0 else 'red' for v in prod_vals]
    axes[1].bar(range(len(prod_labels)), prod_vals, color=colors, alpha=0.7)
    axes[1].set_xticks(range(len(prod_labels)))
    axes[1].set_xticklabels(prod_labels, rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('Correlation')
    axes[1].set_title('Production Mix Correlation')

# 3. Feature importance
top_10_rf = importance_df.head(10)
axes[2].barh(range(len(top_10_rf)), top_10_rf['Importance'], color='coral')
axes[2].set_yticks(range(len(top_10_rf)))
axes[2].set_yticklabels(top_10_rf['Feature'], fontsize=9)
axes[2].set_xlabel('Importance')
axes[2].set_title('Top 10 Features by RF Importance')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'src/analyze' / 'summary_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Generate final report
report_file = PROJECT_ROOT / 'src/analyze' / 'analysis_report.txt'

with open(report_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("EDF ENERGY CONSUMPTION PREDICTION - DATA ANALYSIS REPORT\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"ANALYSIS DATE: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("1. DATASET INFORMATION\n")
    f.write("-"*40 + "\n")
    f.write(f"Total records: {len(df_analysis):,}\n")
    f.write(f"Date range: {df_analysis['datetime'].min()} to {df_analysis['datetime'].max()}\n")
    f.write(f"Total features: {len(df_analysis.columns)}\n")
    f.write(f"Numeric features: {len(df_analysis.select_dtypes(include=[np.number]).columns)}\n\n")
    
    f.write("2. TARGET VARIABLE STATISTICS (Consommation)\n")
    f.write("-"*40 + "\n")
    f.write(f"Mean: {df_analysis['consommation'].mean():.2f} MW\n")
    f.write(f"Median: {df_analysis['consommation'].median():.2f} MW\n")
    f.write(f"Std Dev: {df_analysis['consommation'].std():.2f} MW\n")
    f.write(f"Min: {df_analysis['consommation'].min():.2f} MW\n")
    f.write(f"Max: {df_analysis['consommation'].max():.2f} MW\n\n")
    
    f.write("3. SELECTED FEATURES ({} total)\n".format(len(selected_features)))
    f.write("-"*40 + "\n")
    for i, feature in enumerate(selected_features, 1):
        corr = target_corr.get(feature, 0)
        f.write(f"{i}. {feature:40s} (corr: {corr:+.3f})\n")
    f.write("\n")
    
    f.write("4. FEATURE GROUPS\n")
    f.write("-"*40 + "\n")
    f.write(f"Production: {len(production_features)} features\n")
    f.write(f"Temporal: {len(temporal_features)} features\n")
    f.write(f"Forecasts: {len(forecast_features)} features\n")
    f.write(f"Exchanges: {len(exchange_features)} features\n")
    f.write(f"Storage: {len(storage_features)} features\n\n")
    
    f.write("5. RECOMMENDATIONS\n")
    f.write("-"*40 + "\n")
    f.write("- Use forecast features as primary predictors (prevision_j, prevision_j_1)\n")
    f.write("- Include production mix, especially nuclear power\n")
    f.write("- Add temporal features for daily/seasonal patterns\n")
    f.write("- Consider lag features for time series modeling\n")
    f.write("- Integrate weather data for temperature-dependent demand\n")
    f.write("- Monitor for data quality issues (ND values, missing data)\n\n")
    
    f.write("="*80 + "\n")
    f.write("END OF REPORT\n")
    f.write("="*80 + "\n")

print(f"\nReport saved to: {report_file}")

In [ ]:
# List all generated files
analyze_dir = PROJECT_ROOT / 'src/analyze'
generated_files = list(analyze_dir.glob('*.*'))

print("\nGenerated files:")
for f in sorted(generated_files):
    size = f.stat().st_size
    size_kb = size / 1024
    print(f"- {f.name} ({size_kb:.2f} KB)")